
# Medical LLM Notebook 7 — Attention Hypergraph Construction and Structural Validation

## Objective
Notebook 7 lifts the word-level MedGemma attention representation from an ordinary directed graph to a **weighted overlapping hypergraph**.

Each response word is a vertex. For every target word \(i\), the hyperedge is constructed from the target and its strongest incoming answer-word attention sources:

\[
e_i=\{i\}\cup S_i^{(k)},
\]

where \(S_i^{(k)}\) contains the top-\(k\) non-self source words with positive attention to target \(i\).

Because Notebook 5 stores attention as

\[
A^{(w)}[\text{target},\text{source}],
\]

the hyperedge therefore represents a **higher-order source-to-target attention neighborhood**.

The hyperedge weight is

\[
\omega(e_i)
=
\frac{1}{|S_i^{(k)}|}
\sum_{j\in S_i^{(k)}} A^{(w)}_{ij}.
\]

The primary construction uses \(k=3\), matching the graph sparsification used in Notebook 5.  
Sensitivity analyses use \(k=5\) and \(k=10\).

## Main goals
1. Construct one attention hypergraph for each factual and hallucinated response.
2. Extract incidence-, overlap-, degree-, and spectral-hypergraph features.
3. Control again for response length using cross-fitted normalization.
4. Perform paired factual–hallucinated statistical tests.
5. Test sensitivity to \(k=3,5,10\).
6. Save vertex-level and hyperedge-level outputs for explainability.


In [1]:

# Run on CPU. No MedGemma inference is performed in this notebook.

!pip -q install scipy scikit-learn statsmodels

import os
import glob
import gc
import zipfile
import warnings

import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from google.colab import files

from scipy.stats import (
    wilcoxon,
    rankdata,
    spearmanr
)

from statsmodels.stats.multitest import multipletests

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import SplineTransformer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression

warnings.filterwarnings("ignore")

SEED = 42
PRIMARY_K = 3
K_VALUES = [3, 5, 10]
N_SPLITS = 5

print("Notebook 7 libraries loaded.")
print("Primary k:", PRIMARY_K)
print("Sensitivity k values:", K_VALUES)


Notebook 7 libraries loaded.
Primary k: 3
Sensitivity k values: [3, 5, 10]


## 1. Upload Notebook 5 outputs

In [2]:

uploaded = files.upload()

zip_files = [
    name
    for name in uploaded.keys()
    if name.lower().endswith(".zip")
]

assert len(zip_files) >= 1, (
    "Upload Notebook5_100Pair_Outputs.zip"
)

ZIP_FILE = zip_files[0]

EXTRACT_DIR = "/content/Notebook7_input"
os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_FILE, "r") as z:
    z.extractall(EXTRACT_DIR)

print("Extracted:", ZIP_FILE)


Saving Notebook5_100Pair_Outputs.zip to Notebook5_100Pair_Outputs.zip
Extracted: Notebook5_100Pair_Outputs.zip


## 2. Load and validate Notebook 5 data

In [3]:

response_matches = glob.glob(
    "/content/Notebook7_input/**/"
    "MedHallu_200_Response_Features.csv",
    recursive=True
)

matrix_matches = glob.glob(
    "/content/Notebook7_input/**/"
    "word_attention_matrices",
    recursive=True
)

assert len(response_matches) >= 1
assert len(matrix_matches) >= 1

RESPONSE_FILE = response_matches[0]
MATRIX_DIR = matrix_matches[0]

responses = pd.read_csv(
    RESPONSE_FILE
)

matrix_files = glob.glob(
    os.path.join(
        MATRIX_DIR,
        "*.npz"
    )
)

print(
    "Response table:",
    responses.shape
)

print(
    "Attention matrices:",
    len(matrix_files)
)

assert len(responses) == 200
assert responses["Pair_ID"].nunique() == 100
assert len(matrix_files) == 200
assert responses.isna().sum().sum() == 0

counts = (
    responses
    .groupby("Pair_ID")[
        "Response_Type"
    ]
    .nunique()
)

assert (counts == 2).all()

print(
    "Validation passed:"
    " 100 complete factual–hallucinated pairs."
)


Response table: (200, 21)
Attention matrices: 200
Validation passed: 100 complete factual–hallucinated pairs.



## 3. Hypergraph mathematical representation

For a response with \(n\) word vertices and \(m\) hyperedges, define the binary incidence matrix

\[
H\in\{0,1\}^{n\times m},
\]

where

\[
H_{ve}=
\begin{cases}
1,& v\in e,\\
0,& \text{otherwise}.
\end{cases}
\]

Let \(W_e\) be the diagonal matrix of hyperedge weights. The weighted hyperdegree is

\[
d(v)=\sum_e \omega(e)H_{ve}.
\]

The hyperedge degree is

\[
\delta(e)=\sum_v H_{ve}.
\]

We use the normalized hypergraph Laplacian

\[
L_H
=
I-
D_v^{-1/2}
H
W_e
D_e^{-1}
H^\top
D_v^{-1/2}.
\]

From this representation, Notebook 7 extracts:

- mean and maximum hyperedge attention weight;
- mean and maximum weighted hyperdegree;
- hyperdegree concentration;
- hyperdegree Gini coefficient;
- fraction of overlapping hyperedge pairs;
- mean Jaccard overlap among overlapping hyperedges;
- density of the 2-section induced by hyperedge co-membership;
- second-smallest normalized hypergraph Laplacian eigenvalue;
- normalized hypergraph spectral entropy.

`Hyperedges` and `Mean_Hyperedge_Size` are also saved as construction diagnostics but are not used as primary inferential features.


## 4. Hypergraph helper functions

In [4]:

def gini_nonnegative(values):

    x = np.asarray(
        values,
        dtype=float
    )

    x = x[
        np.isfinite(x)
    ]

    if len(x) == 0:
        return np.nan

    x = np.clip(
        x,
        0,
        None
    )

    total = x.sum()

    if total <= 1e-15:
        return 0.0

    x = np.sort(x)

    n = len(x)

    weighted_sum = np.sum(
        np.arange(
            1,
            n + 1
        ) * x
    )

    gini = (
        2.0
        * weighted_sum
        / (n * total)
        -
        (n + 1.0) / n
    )

    return float(gini)


def build_attention_hypergraph(
    attention,
    words,
    k=3
):

    A = np.asarray(
        attention,
        dtype=float
    )

    words = np.asarray(
        words,
        dtype=str
    )

    n = len(words)

    assert A.shape == (
        n,
        n
    )

    hyperedges = []

    for target in range(n):

        row = A[
            target
        ]

        indices = np.arange(
            n
        )

        valid_sources = indices[
            (indices != target)
            &
            np.isfinite(row)
            &
            (row > 0)
        ]

        # Decoder causal attention means that
        # early response words can have fewer
        # than k valid non-self answer sources.
        if len(valid_sources) == 0:
            continue

        order = valid_sources[
            np.argsort(
                row[
                    valid_sources
                ]
            )[::-1]
        ]

        source_ids = order[
            :min(
                k,
                len(order)
            )
        ]

        if len(source_ids) == 0:
            continue

        source_weights = row[
            source_ids
        ]

        members = np.unique(
            np.concatenate(
                [
                    [target],
                    source_ids
                ]
            )
        ).astype(int)

        if len(members) < 2:
            continue

        hyperedge_weight = float(
            np.mean(
                source_weights
            )
        )

        attention_mass = float(
            np.sum(
                source_weights
            )
        )

        hyperedges.append(
            {
                "anchor_id":
                    int(target),

                "anchor_word":
                    str(
                        words[
                            target
                        ]
                    ),

                "source_ids":
                    source_ids.astype(
                        int
                    ),

                "source_words":
                    [
                        str(
                            words[j]
                        )
                        for j
                        in source_ids
                    ],

                "member_ids":
                    members,

                "member_words":
                    [
                        str(
                            words[j]
                        )
                        for j
                        in members
                    ],

                "weight":
                    hyperedge_weight,

                "attention_mass":
                    attention_mass
            }
        )

    m = len(
        hyperedges
    )

    H = np.zeros(
        (
            n,
            m
        ),
        dtype=float
    )

    weights = np.zeros(
        m,
        dtype=float
    )

    for e_idx, edge in enumerate(
        hyperedges
    ):

        H[
            edge[
                "member_ids"
            ],
            e_idx
        ] = 1.0

        weights[
            e_idx
        ] = edge[
            "weight"
        ]

    return (
        H,
        weights,
        hyperedges
    )


In [5]:

def compute_hypergraph_features(
    H,
    weights
):

    H = np.asarray(
        H,
        dtype=float
    )

    weights = np.asarray(
        weights,
        dtype=float
    )

    n, m = H.shape

    if m == 0:

        features = {
            "Hyperedges": 0,
            "Mean_Hyperedge_Size": 0.0,
            "Mean_Hyperedge_Weight": 0.0,
            "Max_Hyperedge_Weight": 0.0,
            "Mean_Hyperdegree": 0.0,
            "Max_Hyperdegree": 0.0,
            "Hyperdegree_Concentration": 0.0,
            "Hyperdegree_Gini": 0.0,
            "Overlap_Pair_Fraction": 0.0,
            "Mean_Overlap_Jaccard": 0.0,
            "TwoSection_Density": 0.0,
            "Hypergraph_Lambda2": 0.0,
            "Hypergraph_Spectral_Entropy": 0.0
        }

        vertex_degree = np.zeros(
            n,
            dtype=float
        )

        membership_count = np.zeros(
            n,
            dtype=float
        )

        return (
            features,
            vertex_degree,
            membership_count,
            np.array([])
        )

    hyperedge_size = H.sum(
        axis=0
    )

    membership_count = H.sum(
        axis=1
    )

    # Weighted hyperdegree:
    # d(v) = sum_e w(e) H(v,e)
    vertex_degree = (
        H
        *
        weights[
            None,
            :
        ]
    ).sum(
        axis=1
    )

    total_degree = float(
        vertex_degree.sum()
    )

    if total_degree > 1e-15:

        degree_concentration = float(
            vertex_degree.max()
            /
            total_degree
        )

    else:

        degree_concentration = 0.0


    # --------------------------------------------------------
    # Hyperedge overlap
    # --------------------------------------------------------

    member_sets = [
        set(
            np.where(
                H[
                    :,
                    e_idx
                ] > 0
            )[0].tolist()
        )
        for e_idx
        in range(m)
    ]

    total_edge_pairs = (
        m
        *
        (m - 1)
        //
        2
    )

    overlap_count = 0
    overlap_jaccard = []

    for a in range(m):

        for b in range(
            a + 1,
            m
        ):

            intersection = (
                member_sets[a]
                &
                member_sets[b]
            )

            if len(
                intersection
            ) == 0:

                continue

            union = (
                member_sets[a]
                |
                member_sets[b]
            )

            jaccard = (
                len(
                    intersection
                )
                /
                len(
                    union
                )
            )

            overlap_count += 1
            overlap_jaccard.append(
                jaccard
            )

    if total_edge_pairs > 0:

        overlap_fraction = (
            overlap_count
            /
            total_edge_pairs
        )

    else:

        overlap_fraction = 0.0

    if len(
        overlap_jaccard
    ) > 0:

        mean_overlap = float(
            np.mean(
                overlap_jaccard
            )
        )

    else:

        mean_overlap = 0.0


    # --------------------------------------------------------
    # 2-section density
    # --------------------------------------------------------

    co_membership = (
        H
        @
        H.T
    ) > 0

    np.fill_diagonal(
        co_membership,
        False
    )

    existing_pairs = int(
        np.count_nonzero(
            np.triu(
                co_membership,
                k=1
            )
        )
    )

    possible_pairs = (
        n
        *
        (n - 1)
        //
        2
    )

    if possible_pairs > 0:

        two_section_density = (
            existing_pairs
            /
            possible_pairs
        )

    else:

        two_section_density = 0.0


    # --------------------------------------------------------
    # Normalized hypergraph Laplacian
    #
    # L = I -
    # Dv^-1/2 H We De^-1 H^T Dv^-1/2
    # --------------------------------------------------------

    inv_sqrt_dv = np.zeros(
        n,
        dtype=float
    )

    valid_v = (
        vertex_degree
        >
        1e-15
    )

    inv_sqrt_dv[
        valid_v
    ] = (
        1.0
        /
        np.sqrt(
            vertex_degree[
                valid_v
            ]
        )
    )

    inv_de = np.zeros(
        m,
        dtype=float
    )

    valid_e = (
        hyperedge_size
        >
        0
    )

    inv_de[
        valid_e
    ] = (
        1.0
        /
        hyperedge_size[
            valid_e
        ]
    )

    weighted_incidence = (
        H
        *
        (
            weights
            *
            inv_de
        )[
            None,
            :
        ]
    )

    theta = (
        inv_sqrt_dv[
            :,
            None
        ]
        *
        (
            weighted_incidence
            @
            H.T
        )
        *
        inv_sqrt_dv[
            None,
            :
        ]
    )

    # Remove tiny numerical asymmetry.
    theta = (
        theta
        +
        theta.T
    ) / 2.0

    L = (
        np.eye(
            n
        )
        -
        theta
    )

    eigenvalues = np.linalg.eigvalsh(
        L
    )

    eigenvalues = np.clip(
        eigenvalues,
        0,
        None
    )

    eigenvalues = np.sort(
        eigenvalues
    )

    if n >= 2:

        lambda2 = float(
            eigenvalues[
                1
            ]
        )

    else:

        lambda2 = 0.0

    eigen_sum = float(
        eigenvalues.sum()
    )

    if (
        eigen_sum > 1e-15
        and
        n > 1
    ):

        probabilities = (
            eigenvalues
            /
            eigen_sum
        )

        probabilities = probabilities[
            probabilities
            >
            1e-15
        ]

        spectral_entropy = float(
            -
            np.sum(
                probabilities
                *
                np.log(
                    probabilities
                )
            )
            /
            np.log(
                n
            )
        )

    else:

        spectral_entropy = 0.0


    features = {

        "Hyperedges":
            int(m),

        "Mean_Hyperedge_Size":
            float(
                np.mean(
                    hyperedge_size
                )
            ),

        "Mean_Hyperedge_Weight":
            float(
                np.mean(
                    weights
                )
            ),

        "Max_Hyperedge_Weight":
            float(
                np.max(
                    weights
                )
            ),

        "Mean_Hyperdegree":
            float(
                np.mean(
                    vertex_degree
                )
            ),

        "Max_Hyperdegree":
            float(
                np.max(
                    vertex_degree
                )
            ),

        "Hyperdegree_Concentration":
            degree_concentration,

        "Hyperdegree_Gini":
            gini_nonnegative(
                vertex_degree
            ),

        "Overlap_Pair_Fraction":
            float(
                overlap_fraction
            ),

        "Mean_Overlap_Jaccard":
            mean_overlap,

        "TwoSection_Density":
            float(
                two_section_density
            ),

        "Hypergraph_Lambda2":
            lambda2,

        "Hypergraph_Spectral_Entropy":
            spectral_entropy
    }

    return (
        features,
        vertex_degree,
        membership_count,
        eigenvalues
    )


## 5. Sanity-check one response

In [6]:

example_pair = "MEDH_096"
example_type = "Factual"

example_file = os.path.join(
    MATRIX_DIR,
    f"{example_pair}_{example_type}.npz"
)

example = np.load(
    example_file,
    allow_pickle=False
)

A_example = example[
    "attention"
]

words_example = example[
    "words"
]

H_example, w_example, edges_example = (
    build_attention_hypergraph(
        A_example,
        words_example,
        k=PRIMARY_K
    )
)

example_features, example_degree, example_memberships, example_eigs = (
    compute_hypergraph_features(
        H_example,
        w_example
    )
)

print(
    "Words:",
    len(
        words_example
    )
)

print(
    "Hyperedges:",
    H_example.shape[
        1
    ]
)

print(
    "Incidence matrix shape:",
    H_example.shape
)

display(
    pd.DataFrame(
        [example_features]
    ).T.rename(
        columns={
            0:
            "Value"
        }
    )
)


Words: 24
Hyperedges: 23
Incidence matrix shape: (24, 23)


,Value
Hyperedges,23.000000
Mean_Hyperedge_Size,3.869565
Mean_Hyperedge_Weight,0.032910
Max_Hyperedge_Weight,0.052683
Mean_Hyperdegree,0.122525
Max_Hyperdegree,0.257025
Hyperdegree_Concentration,0.087406
Hyperdegree_Gini,0.247901
Overlap_Pair_Fraction,0.395257
Mean_Overlap_Jaccard,0.280071


## 6. Extract hypergraph features for all 200 responses and \(k=3,5,10\)

In [7]:

feature_rows = []

primary_edge_rows = []
primary_vertex_rows = []

for _, meta in tqdm(
    responses.iterrows(),
    total=len(responses),
    desc="Building attention hypergraphs"
):

    pair_id = str(
        meta[
            "Pair_ID"
        ]
    )

    response_type = str(
        meta[
            "Response_Type"
        ]
    )

    matrix_file = os.path.join(
        MATRIX_DIR,
        f"{pair_id}_{response_type}.npz"
    )

    data = np.load(
        matrix_file,
        allow_pickle=False
    )

    attention = data[
        "attention"
    ]

    words = data[
        "words"
    ]

    assert attention.shape == (
        len(words),
        len(words)
    )

    assert np.isfinite(
        attention
    ).all()

    for k in K_VALUES:

        H, hyperedge_weights, hyperedges = (
            build_attention_hypergraph(
                attention,
                words,
                k=k
            )
        )

        (
            features,
            vertex_degree,
            membership_count,
            eigenvalues
        ) = compute_hypergraph_features(
            H,
            hyperedge_weights
        )

        record = {

            "Pair_ID":
                pair_id,

            "Response_Type":
                response_type,

            "Label":
                int(
                    meta[
                        "Label"
                    ]
                ),

            "Difficulty":
                meta[
                    "Difficulty"
                ],

            "Hallucination_Category":
                meta[
                    "Hallucination_Category"
                ],

            "Words":
                int(
                    meta[
                        "Words"
                    ]
                ),

            "K":
                int(k),

            **features
        }

        feature_rows.append(
            record
        )


        # Detailed explainability outputs are
        # saved for the primary k only.
        if k == PRIMARY_K:

            total_degree = float(
                vertex_degree.sum()
            )

            for node_id, word in enumerate(
                words
            ):

                if total_degree > 1e-15:

                    degree_share = (
                        vertex_degree[
                            node_id
                        ]
                        /
                        total_degree
                    )

                else:

                    degree_share = 0.0

                primary_vertex_rows.append(
                    {
                        "Pair_ID":
                            pair_id,

                        "Response_Type":
                            response_type,

                        "Node_ID":
                            int(
                                node_id
                            ),

                        "Word":
                            str(
                                word
                            ),

                        "Weighted_Hyperdegree":
                            float(
                                vertex_degree[
                                    node_id
                                ]
                            ),

                        "Hyperdegree_Share":
                            float(
                                degree_share
                            ),

                        "Membership_Count":
                            int(
                                membership_count[
                                    node_id
                                ]
                            )
                    }
                )

            for edge_id, edge in enumerate(
                hyperedges
            ):

                primary_edge_rows.append(
                    {
                        "Pair_ID":
                            pair_id,

                        "Response_Type":
                            response_type,

                        "Hyperedge_ID":
                            int(
                                edge_id
                            ),

                        "Anchor_ID":
                            int(
                                edge[
                                    "anchor_id"
                                ]
                            ),

                        "Anchor_Word":
                            edge[
                                "anchor_word"
                            ],

                        "Source_IDs":
                            "|".join(
                                map(
                                    str,
                                    edge[
                                        "source_ids"
                                    ]
                                )
                            ),

                        "Source_Words":
                            " | ".join(
                                edge[
                                    "source_words"
                                ]
                            ),

                        "Member_IDs":
                            "|".join(
                                map(
                                    str,
                                    edge[
                                        "member_ids"
                                    ]
                                )
                            ),

                        "Member_Words":
                            " | ".join(
                                edge[
                                    "member_words"
                                ]
                            ),

                        "Hyperedge_Size":
                            int(
                                len(
                                    edge[
                                        "member_ids"
                                    ]
                                )
                            ),

                        "Hyperedge_Weight":
                            float(
                                edge[
                                    "weight"
                                ]
                            ),

                        "Attention_Mass":
                            float(
                                edge[
                                    "attention_mass"
                                ]
                            )
                    }
                )

    del data
    gc.collect()


hypergraph_features = pd.DataFrame(
    feature_rows
)

primary_hyperedges = pd.DataFrame(
    primary_edge_rows
)

primary_vertices = pd.DataFrame(
    primary_vertex_rows
)

print(
    "Hypergraph feature rows:",
    len(
        hypergraph_features
    )
)

print(
    "Primary hyperedge rows:",
    len(
        primary_hyperedges
    )
)

print(
    "Primary vertex rows:",
    len(
        primary_vertices
    )
)

display(
    hypergraph_features.head()
)


Building attention hypergraphs:   0%|          | 0/200 [00:00<?, ?it/s]

Hypergraph feature rows: 600
Primary hyperedge rows: 6351
Primary vertex rows: 6551


,Pair_ID,Response_Type,Label,Difficulty,Hallucination_Category,Words,K,Hyperedges,Mean_Hyperedge_Size,Mean_Hyperedge_Weight,Max_Hyperedge_Weight,Mean_Hyperdegree,Max_Hyperdegree,Hyperdegree_Concentration,Hyperdegree_Gini,Overlap_Pair_Fraction,Mean_Overlap_Jaccard,TwoSection_Density,Hypergraph_Lambda2,Hypergraph_Spectral_Entropy
0,MEDH_001,Factual,0,easy,Incomplete Information,34,3,33,3.909091,0.034685,0.061701,0.130334,0.474694,0.107121,0.343396,0.443182,0.227869,0.196078,0.068945,0.969386
1,MEDH_001,Factual,0,easy,Incomplete Information,34,5,33,5.696970,0.027365,0.061701,0.147471,0.501714,0.100062,0.378295,0.662879,0.239964,0.336898,0.148425,0.979881
2,MEDH_001,Factual,0,easy,Incomplete Information,34,10,33,9.636364,0.019582,0.061701,0.170222,0.489311,0.084546,0.382730,0.848485,0.337688,0.590018,0.306749,0.987081
3,MEDH_001,Hallucinated,1,easy,Incomplete Information,18,3,17,3.823529,0.046059,0.184113,0.149462,0.622899,0.231535,0.393811,0.742647,0.256554,0.339869,0.225280,0.959334
4,MEDH_001,Hallucinated,1,easy,Incomplete Information,18,5,17,5.411765,0.038068,0.184113,0.158390,0.590711,0.207193,0.419077,0.948529,0.282898,0.575163,0.394690,0.971266


## 7. Validate extracted hypergraphs

In [8]:

assert len(
    hypergraph_features
) == (
    200
    *
    len(
        K_VALUES
    )
)

assert hypergraph_features[
    [
        "Pair_ID",
        "Response_Type",
        "K"
    ]
].duplicated().sum() == 0

assert hypergraph_features.isna().sum().sum() == 0

for k in K_VALUES:

    subset = hypergraph_features[
        hypergraph_features[
            "K"
        ] == k
    ]

    assert len(
        subset
    ) == 200

    assert subset[
        "Pair_ID"
    ].nunique() == 100

    assert (
        subset[
            "Response_Type"
        ].value_counts().to_dict()
        ==
        {
            "Factual": 100,
            "Hallucinated": 100
        }
    )

print(
    "Hypergraph extraction validation passed."
)


Hypergraph extraction validation passed.



## 8. Primary inferential hypergraph features

Construction diagnostics such as the number of hyperedges and mean hyperedge size are saved but excluded from primary inferential testing because fixed-\(k\) construction directly constrains them.

The following features are tested:


In [9]:

HYPER_FEATURES = [

    "Mean_Hyperedge_Weight",

    "Max_Hyperedge_Weight",

    "Mean_Hyperdegree",

    "Max_Hyperdegree",

    "Hyperdegree_Concentration",

    "Hyperdegree_Gini",

    "Overlap_Pair_Fraction",

    "Mean_Overlap_Jaccard",

    "TwoSection_Density",

    "Hypergraph_Lambda2",

    "Hypergraph_Spectral_Entropy"
]

print(
    "Inferential hypergraph features:",
    len(
        HYPER_FEATURES
    )
)


Inferential hypergraph features: 11


## 9. Raw length dependence before normalization

In [ ]:

raw_length_rows = []

for k in K_VALUES:

    subset = hypergraph_features[
        hypergraph_features[
            "K"
        ] == k
    ]

    for feature in HYPER_FEATURES:

        rho, p = spearmanr(
            subset[
                "Words"
            ],
            subset[
                feature
            ]
        )

        raw_length_rows.append(
            {
                "K":
                    int(k),

                "Feature":
                    feature,

                "Raw_rho_with_Words":
                    rho,

                "Raw_p":
                    p
            }
        )

raw_length_dependence = pd.DataFrame(
    raw_length_rows
)

display(
    raw_length_dependence[
        raw_length_dependence[
            "K"
        ] == PRIMARY_K
    ]
    .sort_values(
        "Raw_rho_with_Words",
        key=lambda s: s.abs(),
        ascending=False
    )
    .round(6)
)



## 10. Cross-fitted response-length normalization

As in Notebook 6B, each hypergraph feature is normalized using only response length:

\[
X^{*}
=
\frac{
X-\widehat{f}(\log(1+n))
}{
s_{\mathrm{train}}
}.
\]

Five-fold `GroupKFold` is performed by `Pair_ID`, so the factual and hallucinated responses from the same medical question remain in the same fold.

The factual/hallucinated label is never used to fit the normalization model.


In [10]:

def crossfit_length_normalization(
    df,
    features,
    model_type="spline",
    n_splits=5
):

    work = (
        df
        .copy()
        .reset_index(
            drop=True
        )
    )

    x = np.log1p(
        work[
            "Words"
        ]
        .astype(float)
        .to_numpy()
    ).reshape(
        -1,
        1
    )

    groups = (
        work[
            "Pair_ID"
        ]
        .astype(str)
        .to_numpy()
    )

    output = work[
        [
            "Pair_ID",
            "Response_Type",
            "Label",
            "Difficulty",
            "Hallucination_Category",
            "Words",
            "K"
        ]
    ].copy()

    fold_id = np.full(
        len(work),
        -1,
        dtype=int
    )

    splitter = GroupKFold(
        n_splits=n_splits
    )

    for feature in features:

        y = (
            work[
                feature
            ]
            .astype(float)
            .to_numpy()
        )

        prediction = np.full(
            len(work),
            np.nan,
            dtype=float
        )

        normalized = np.full(
            len(work),
            np.nan,
            dtype=float
        )

        for fold, (
            train_idx,
            test_idx
        ) in enumerate(
            splitter.split(
                x,
                y,
                groups=groups
            )
        ):

            if model_type == "spline":

                model = make_pipeline(
                    SplineTransformer(
                        n_knots=4,
                        degree=3,
                        include_bias=False
                    ),
                    LinearRegression()
                )

            elif model_type == "linear":

                model = (
                    LinearRegression()
                )

            else:

                raise ValueError(
                    "model_type must be "
                    "'spline' or 'linear'"
                )

            model.fit(
                x[
                    train_idx
                ],
                y[
                    train_idx
                ]
            )

            pred_train = model.predict(
                x[
                    train_idx
                ]
            )

            pred_test = model.predict(
                x[
                    test_idx
                ]
            )

            train_residual = (
                y[
                    train_idx
                ]
                -
                pred_train
            )

            train_scale = np.std(
                train_residual,
                ddof=1
            )

            if (
                not np.isfinite(
                    train_scale
                )
                or
                train_scale <= 1e-12
            ):

                train_scale = 1.0

            prediction[
                test_idx
            ] = pred_test

            normalized[
                test_idx
            ] = (
                y[
                    test_idx
                ]
                -
                pred_test
            ) / train_scale

            fold_id[
                test_idx
            ] = fold

        output[
            f"{feature}_Predicted_From_Length"
        ] = prediction

        output[
            f"{feature}_LengthNorm"
        ] = normalized

    output[
        "Fold"
    ] = fold_id

    return output


In [11]:

normalized_by_k = []

for k in K_VALUES:

    subset = (
        hypergraph_features[
            hypergraph_features[
                "K"
            ] == k
        ]
        .copy()
    )

    normalized_subset = (
        crossfit_length_normalization(
            subset,
            HYPER_FEATURES,
            model_type="spline",
            n_splits=N_SPLITS
        )
    )

    normalized_by_k.append(
        normalized_subset
    )

normalized_hypergraphs = pd.concat(
    normalized_by_k,
    ignore_index=True
)

assert (
    normalized_hypergraphs
    .filter(
        like="_LengthNorm"
    )
    .isna()
    .sum()
    .sum()
    ==
    0
)

print(
    "Spline length normalization complete."
)


Spline length normalization complete.


## 11. Confirm residual length dependence after normalization

In [14]:
# ============================================================
# CELL 12 — LENGTH DEPENDENCE DIAGNOSTICS
# Self-contained version
# ============================================================

import pandas as pd
from scipy.stats import spearmanr


# ------------------------------------------------------------
# 1. Recalculate RAW length dependence
# ------------------------------------------------------------

raw_length_rows = []

for k in K_VALUES:

    subset = hypergraph_features[
        hypergraph_features["K"] == k
    ]

    for feature in HYPER_FEATURES:

        rho, p = spearmanr(
            subset["Words"],
            subset[feature]
        )

        raw_length_rows.append(
            {
                "K": int(k),
                "Feature": feature,
                "Raw_rho_with_Words": rho,
                "Raw_p": p
            }
        )


raw_length_dependence = pd.DataFrame(
    raw_length_rows
)


# ------------------------------------------------------------
# 2. Calculate NORMALIZED length dependence
# ------------------------------------------------------------

normalized_length_rows = []

for k in K_VALUES:

    subset = normalized_hypergraphs[
        normalized_hypergraphs["K"] == k
    ]

    for feature in HYPER_FEATURES:

        rho, p = spearmanr(
            subset["Words"],
            subset[
                f"{feature}_LengthNorm"
            ]
        )

        normalized_length_rows.append(
            {
                "K": int(k),
                "Feature": feature,
                "Normalized_rho_with_Words": rho,
                "Normalized_p": p
            }
        )


normalized_length_dependence = pd.DataFrame(
    normalized_length_rows
)


# ------------------------------------------------------------
# 3. Merge raw and normalized results
# ------------------------------------------------------------

length_diagnostics = (
    raw_length_dependence
    .merge(
        normalized_length_dependence,
        on=[
            "K",
            "Feature"
        ]
    )
)


# ------------------------------------------------------------
# 4. Measure reduction in length dependence
# ------------------------------------------------------------

length_diagnostics[
    "Abs_Rho_Reduction"
] = (
    length_diagnostics[
        "Raw_rho_with_Words"
    ].abs()
    -
    length_diagnostics[
        "Normalized_rho_with_Words"
    ].abs()
)


# ------------------------------------------------------------
# 5. Show primary k = 3 results
# ------------------------------------------------------------

primary_length_diagnostics = (
    length_diagnostics[
        length_diagnostics[
            "K"
        ] == PRIMARY_K
    ]
    .sort_values(
        "Abs_Rho_Reduction",
        ascending=False
    )
    .reset_index(drop=True)
)


print(
    "Length diagnostics completed."
)

print(
    "Total diagnostic rows:",
    len(length_diagnostics)
)

print(
    "Primary k:",
    PRIMARY_K
)

display(
    primary_length_diagnostics.round(6)
)

Length diagnostics completed.
Total diagnostic rows: 33
Primary k: 3


,K,Feature,Raw_rho_with_Words,Raw_p,Normalized_rho_with_Words,Normalized_p,Abs_Rho_Reduction
0,3,TwoSection_Density,-0.992363,0.000000,0.017409,0.806711,0.974955
1,3,Hypergraph_Spectral_Entropy,0.946925,0.000000,-0.001256,0.985913,0.945668
2,3,Overlap_Pair_Fraction,-0.880995,0.000000,0.074901,0.291833,0.806093
3,3,Mean_Overlap_Jaccard,-0.798646,0.000000,0.026921,0.705129,0.771725
4,3,Hyperdegree_Concentration,-0.741250,0.000000,0.058427,0.411186,0.682823
5,3,Hypergraph_Lambda2,-0.616758,0.000000,0.112952,0.111280,0.503806
6,3,Max_Hyperdegree,0.567221,0.000000,-0.072233,0.309410,0.494988
7,3,Mean_Hyperedge_Weight,-0.356006,0.000000,-0.004132,0.953691,0.351874
8,3,Hyperdegree_Gini,0.254031,0.000284,0.006410,0.928220,0.247621
9,3,Mean_Hyperdegree,0.142605,0.043967,-0.004047,0.954640,0.138557


## 12. Paired normalized factual–hallucinated tests

In [15]:

def signed_rank_biserial(
    delta
):

    delta = np.asarray(
        delta,
        dtype=float
    )

    delta = delta[
        np.isfinite(
            delta
        )
    ]

    delta = delta[
        delta != 0
    ]

    if len(
        delta
    ) == 0:

        return 0.0

    ranks = rankdata(
        np.abs(
            delta
        )
    )

    positive = ranks[
        delta > 0
    ].sum()

    negative = ranks[
        delta < 0
    ].sum()

    denominator = (
        positive
        +
        negative
    )

    if denominator <= 0:
        return 0.0

    return float(
        (
            positive
            -
            negative
        )
        /
        denominator
    )


def bootstrap_mean_ci(
    values,
    B=5000,
    seed=42
):

    x = np.asarray(
        values,
        dtype=float
    )

    rng = np.random.default_rng(
        seed
    )

    boot = np.empty(
        B,
        dtype=float
    )

    for b in range(B):

        sample = rng.choice(
            x,
            size=len(x),
            replace=True
        )

        boot[b] = np.mean(
            sample
        )

    return np.quantile(
        boot,
        [
            0.025,
            0.975
        ]
    )


In [16]:

paired_test_rows = []
paired_normalized_rows = []

for k in K_VALUES:

    subset = normalized_hypergraphs[
        normalized_hypergraphs[
            "K"
        ] == k
    ]

    factual = (
        subset[
            subset[
                "Response_Type"
            ] == "Factual"
        ]
        .set_index(
            "Pair_ID"
        )
    )

    hallucinated = (
        subset[
            subset[
                "Response_Type"
            ] == "Hallucinated"
        ]
        .set_index(
            "Pair_ID"
        )
    )

    pair_ids = sorted(
        set(
            factual.index
        )
        &
        set(
            hallucinated.index
        )
    )

    assert len(
        pair_ids
    ) == 100

    p_values = []
    temporary_rows = []

    for feature in HYPER_FEATURES:

        column = (
            f"{feature}_LengthNorm"
        )

        f = (
            factual.loc[
                pair_ids,
                column
            ]
            .astype(float)
            .to_numpy()
        )

        h = (
            hallucinated.loc[
                pair_ids,
                column
            ]
            .astype(float)
            .to_numpy()
        )

        delta = (
            h
            -
            f
        )

        if np.allclose(
            delta,
            0
        ):

            statistic = 0.0
            p_value = 1.0

        else:

            test = wilcoxon(
                h,
                f,
                alternative="two-sided",
                zero_method="wilcox",
                method="auto"
            )

            statistic = float(
                test.statistic
            )

            p_value = float(
                test.pvalue
            )

        ci_low, ci_high = (
            bootstrap_mean_ci(
                delta,
                B=5000,
                seed=SEED
                +
                int(k)
            )
        )

        sd_delta = np.std(
            delta,
            ddof=1
        )

        temporary_rows.append(
            {
                "K":
                    int(k),

                "Feature":
                    feature,

                "Factual_Norm_Mean":
                    float(
                        np.mean(
                            f
                        )
                    ),

                "Hallucinated_Norm_Mean":
                    float(
                        np.mean(
                            h
                        )
                    ),

                "Mean_Normalized_Delta":
                    float(
                        np.mean(
                            delta
                        )
                    ),

                "Mean_Delta_CI_Low":
                    float(
                        ci_low
                    ),

                "Mean_Delta_CI_High":
                    float(
                        ci_high
                    ),

                "Median_Normalized_Delta":
                    float(
                        np.median(
                            delta
                        )
                    ),

                "Wilcoxon_W":
                    statistic,

                "Wilcoxon_p":
                    p_value,

                "Rank_Biserial":
                    signed_rank_biserial(
                        delta
                    ),

                "Cohen_dz":
                    float(
                        np.mean(
                            delta
                        )
                        /
                        sd_delta
                    )
                    if sd_delta > 1e-15
                    else 0.0,

                "Positive_Delta_n":
                    int(
                        (
                            delta
                            >
                            0
                        ).sum()
                    ),

                "Negative_Delta_n":
                    int(
                        (
                            delta
                            <
                            0
                        ).sum()
                    ),

                "Zero_Delta_n":
                    int(
                        (
                            delta
                            ==
                            0
                        ).sum()
                    )
            }
        )

        p_values.append(
            p_value
        )

        # Save pair-level normalized differences.
        for idx, pair_id in enumerate(
            pair_ids
        ):

            paired_normalized_rows.append(
                {
                    "K":
                        int(k),

                    "Pair_ID":
                        pair_id,

                    "Feature":
                        feature,

                    "Factual_Norm":
                        float(
                            f[
                                idx
                            ]
                        ),

                    "Hallucinated_Norm":
                        float(
                            h[
                                idx
                            ]
                        ),

                    "DeltaNorm":
                        float(
                            delta[
                                idx
                            ]
                        )
                }
            )

    corrected = multipletests(
        p_values,
        method="holm"
    )[1]

    for row, adjusted_p in zip(
        temporary_rows,
        corrected
    ):

        row[
            "Wilcoxon_p_Holm"
        ] = float(
            adjusted_p
        )

        row[
            "Significant_Holm_0.05"
        ] = bool(
            adjusted_p
            <
            0.05
        )

        paired_test_rows.append(
            row
        )


paired_hypergraph_tests = pd.DataFrame(
    paired_test_rows
)

paired_normalized_differences = pd.DataFrame(
    paired_normalized_rows
)

display(
    paired_hypergraph_tests[
        paired_hypergraph_tests[
            "K"
        ] == PRIMARY_K
    ]
    .sort_values(
        "Wilcoxon_p_Holm"
    )
    .round(6)
)


,K,Feature,Factual_Norm_Mean,Hallucinated_Norm_Mean,Mean_Normalized_Delta,Mean_Delta_CI_Low,Mean_Delta_CI_High,Median_Normalized_Delta,Wilcoxon_W,Wilcoxon_p,Rank_Biserial,Cohen_dz,Positive_Delta_n,Negative_Delta_n,Zero_Delta_n,Wilcoxon_p_Holm,Significant_Holm_0.05
6,3,Overlap_Pair_Fraction,0.206557,-0.225494,-0.432051,-0.686840,-0.185821,-0.457154,1497.0,0.000408,-0.407129,-0.338015,37,63,0,0.004492,True
5,3,Hyperdegree_Gini,0.206404,-0.223850,-0.430254,-0.695525,-0.174917,-0.484316,1669.0,0.003248,-0.339010,-0.317049,43,57,0,0.032483,True
7,3,Mean_Overlap_Jaccard,-0.171257,0.192216,0.363472,0.100180,0.633895,0.214436,1853.0,0.020857,0.266139,0.263272,57,43,0,0.187717,False
4,3,Hyperdegree_Concentration,0.122848,-0.137786,-0.260633,-0.514367,-0.022157,-0.330227,1855.0,0.021241,-0.265347,-0.205705,40,60,0,0.187717,False
10,3,Hypergraph_Spectral_Entropy,0.144233,-0.162461,-0.306694,-0.577583,-0.043855,-0.382710,1857.0,0.021630,-0.264554,-0.219820,39,61,0,0.187717,False
3,3,Max_Hyperdegree,0.127167,-0.143909,-0.271076,-0.540654,-0.013842,-0.189017,1976.0,0.059074,-0.217426,-0.200528,43,57,0,0.354446,False
9,3,Hypergraph_Lambda2,0.088175,-0.095667,-0.183842,-0.448957,0.078973,-0.247441,1996.0,0.068931,-0.209505,-0.134520,37,63,0,0.354446,False
1,3,Max_Hyperedge_Weight,-0.133692,0.151283,0.284975,0.019435,0.562895,0.136275,1977.0,0.082182,0.201212,0.199322,60,39,1,0.354446,False
0,3,Mean_Hyperedge_Weight,-0.073452,0.074322,0.147773,-0.103196,0.409234,0.059677,2349.0,0.545083,0.069703,0.112193,55,45,0,1.000000,False
2,3,Mean_Hyperdegree,-0.049649,0.049561,0.099211,-0.134740,0.346639,0.006289,2476.0,0.866207,0.019406,0.079934,50,50,0,1.000000,False


## 13. Paired normalized differences versus paired word-count difference

In [ ]:

word_delta = (
    responses.pivot(
        index="Pair_ID",
        columns="Response_Type",
        values="Words"
    )
)

word_delta[
    "Delta_Words"
] = (
    word_delta[
        "Hallucinated"
    ]
    -
    word_delta[
        "Factual"
    ]
)

delta_length_rows = []

for k in K_VALUES:

    subset = paired_normalized_differences[
        paired_normalized_differences[
            "K"
        ] == k
    ]

    for feature in HYPER_FEATURES:

        feature_delta = (
            subset[
                subset[
                    "Feature"
                ] == feature
            ]
            .set_index(
                "Pair_ID"
            )[
                "DeltaNorm"
            ]
        )

        aligned = (
            pd.concat(
                [
                    feature_delta,
                    word_delta[
                        "Delta_Words"
                    ]
                ],
                axis=1,
                join="inner"
            )
            .dropna()
        )

        rho, p = spearmanr(
            aligned[
                "DeltaNorm"
            ],
            aligned[
                "Delta_Words"
            ]
        )

        delta_length_rows.append(
            {
                "K":
                    int(k),

                "Feature":
                    feature,

                "rho_DeltaNorm_vs_DeltaWords":
                    rho,

                "p":
                    p
            }
        )

delta_length_diagnostics = pd.DataFrame(
    delta_length_rows
)

for k in K_VALUES:

    mask = (
        delta_length_diagnostics[
            "K"
        ] == k
    )

    delta_length_diagnostics.loc[
        mask,
        "p_Holm"
    ] = multipletests(
        delta_length_diagnostics.loc[
            mask,
            "p"
        ],
        method="holm"
    )[1]

display(
    delta_length_diagnostics[
        delta_length_diagnostics[
            "K"
        ] == PRIMARY_K
    ]
    .sort_values(
        "rho_DeltaNorm_vs_DeltaWords",
        key=lambda s: s.abs(),
        ascending=False
    )
    .round(6)
)



## 14. Sensitivity to hyperedge neighborhood size \(k\)

A feature is **direction-stable** when the sign of the mean normalized hallucinated-minus-factual difference is the same for \(k=3,5,10\).

The notebook reports two separate robustness flags:

- `Significant_All_K`: Holm-adjusted \(p<0.05\) at all three \(k\) values.
- `Primary_Signal_Stable_Direction`: significant at the preregistered primary \(k=3\) and direction-stable across \(k=3,5,10\).

These are intentionally kept separate so sensitivity results are not overstated.


In [17]:

sensitivity_rows = []

for feature in HYPER_FEATURES:

    sub = (
        paired_hypergraph_tests[
            paired_hypergraph_tests[
                "Feature"
            ] == feature
        ]
        .set_index(
            "K"
        )
    )

    row = {
        "Feature":
            feature
    }

    directions = []
    significance = []

    for k in K_VALUES:

        delta = float(
            sub.loc[
                k,
                "Mean_Normalized_Delta"
            ]
        )

        adjusted_p = float(
            sub.loc[
                k,
                "Wilcoxon_p_Holm"
            ]
        )

        rank_biserial = float(
            sub.loc[
                k,
                "Rank_Biserial"
            ]
        )

        row[
            f"Mean_Delta_K{k}"
        ] = delta

        row[
            f"Rank_Biserial_K{k}"
        ] = rank_biserial

        row[
            f"Holm_p_K{k}"
        ] = adjusted_p

        directions.append(
            np.sign(
                delta
            )
        )

        significance.append(
            adjusted_p < 0.05
        )

    nonzero_directions = [
        d
        for d in directions
        if d != 0
    ]

    row[
        "Same_Direction_All_K"
    ] = (
        len(
            nonzero_directions
        ) == len(
            K_VALUES
        )
        and
        len(
            set(
                nonzero_directions
            )
        ) == 1
    )

    row[
        "Significant_All_K"
    ] = all(
        significance
    )

    row[
        "Primary_K3_Significant"
    ] = significance[
        K_VALUES.index(
            PRIMARY_K
        )
    ]

    row[
        "Primary_Signal_Stable_Direction"
    ] = (
        row[
            "Primary_K3_Significant"
        ]
        and
        row[
            "Same_Direction_All_K"
        ]
    )

    sensitivity_rows.append(
        row
    )


k_sensitivity = pd.DataFrame(
    sensitivity_rows
)

display(
    k_sensitivity
    .sort_values(
        [
            "Significant_All_K",
            "Primary_Signal_Stable_Direction",
            "Holm_p_K3"
        ],
        ascending=[
            False,
            False,
            True
        ]
    )
    .round(6)
)


,Feature,Mean_Delta_K3,Rank_Biserial_K3,Holm_p_K3,Mean_Delta_K5,Rank_Biserial_K5,Holm_p_K5,Mean_Delta_K10,Rank_Biserial_K10,Holm_p_K10,Same_Direction_All_K,Significant_All_K,Primary_K3_Significant,Primary_Signal_Stable_Direction
6,Overlap_Pair_Fraction,-0.432051,-0.407129,0.004492,-0.348184,-0.293465,0.119246,-0.141490,-0.076040,1.000000,True,False,True,True
5,Hyperdegree_Gini,-0.430254,-0.339010,0.032483,-0.317977,-0.222970,0.476055,-0.154993,-0.125941,1.000000,True,False,True,True
4,Hyperdegree_Concentration,-0.260633,-0.265347,0.187717,-0.215471,-0.222178,0.476055,-0.005329,0.004752,1.000000,True,False,False,False
7,Mean_Overlap_Jaccard,0.363472,0.266139,0.187717,0.132799,0.152871,0.922224,-0.096760,-0.094653,1.000000,False,False,False,False
10,Hypergraph_Spectral_Entropy,-0.306694,-0.264554,0.187717,-0.160528,-0.148515,0.922224,-0.065493,-0.054257,1.000000,True,False,False,False
1,Max_Hyperedge_Weight,0.284975,0.201212,0.354446,0.268262,0.195960,0.569053,0.254380,0.195960,0.904953,True,False,False,False
3,Max_Hyperdegree,-0.271076,-0.217426,0.354446,-0.247823,-0.200792,0.569053,-0.081077,-0.051881,1.000000,True,False,False,False
9,Hypergraph_Lambda2,-0.183842,-0.209505,0.354446,-0.139305,-0.129505,0.922224,-0.095694,-0.099802,1.000000,True,False,False,False
0,Mean_Hyperedge_Weight,0.147773,0.069703,1.000000,0.131642,0.059010,1.000000,0.133540,0.067327,1.000000,True,False,False,False
2,Mean_Hyperdegree,0.099211,0.019406,1.000000,0.044325,-0.022574,1.000000,0.013301,-0.009109,1.000000,True,False,False,False



## 15. Linear-normalization sensitivity for the primary \(k=3\)

Notebook 6B showed that the normalization model itself should also be checked.  
Here, the primary \(k=3\) hypergraph analysis is repeated using a simple linear model of \(\log(1+\text{Words})\).


In [19]:

primary_raw = (
    hypergraph_features[
        hypergraph_features[
            "K"
        ] == PRIMARY_K
    ]
    .copy()
)

primary_linear = (
    crossfit_length_normalization(
        primary_raw,
        HYPER_FEATURES,
        model_type="linear",
        n_splits=N_SPLITS
    )
)

f_linear = (
    primary_linear[
        primary_linear[
            "Response_Type"
        ] == "Factual"
    ]
    .set_index(
        "Pair_ID"
    )
)

h_linear = (
    primary_linear[
        primary_linear[
            "Response_Type"
        ] == "Hallucinated"
    ]
    .set_index(
        "Pair_ID"
    )
)

pair_ids = sorted(
    set(
        f_linear.index
    )
    &
    set(
        h_linear.index
    )
)

linear_rows = []
linear_p = []

for feature in HYPER_FEATURES:

    column = (
        f"{feature}_LengthNorm"
    )

    f = (
        f_linear.loc[
            pair_ids,
            column
        ]
        .astype(float)
        .to_numpy()
    )

    h = (
        h_linear.loc[
            pair_ids,
            column
        ]
        .astype(float)
        .to_numpy()
    )

    delta = h - f

    if np.allclose(
        delta,
        0
    ):

        p_value = 1.0

    else:

        p_value = wilcoxon(
            h,
            f,
            alternative="two-sided",
            zero_method="wilcox",
            method="auto"
        ).pvalue

    linear_rows.append(
        {
            "Feature":
                feature,

            "Mean_Delta_Linear":
                float(
                    np.mean(
                        delta
                    )
                ),

            "Rank_Biserial_Linear":
                signed_rank_biserial(
                    delta
                ),

            "Wilcoxon_p_Linear":
                float(
                    p_value
                )
        }
    )

    linear_p.append(
        p_value
    )

linear_corrected = multipletests(
    linear_p,
    method="holm"
)[1]

linear_normalization_sensitivity = pd.DataFrame(
    linear_rows
)

linear_normalization_sensitivity[
    "Wilcoxon_p_Linear_Holm"
] = linear_corrected

linear_normalization_sensitivity[
    "Significant_Linear_Holm_0.05"
] = (
    linear_normalization_sensitivity[
        "Wilcoxon_p_Linear_Holm"
    ]
    <
    0.05
)

display(
    linear_normalization_sensitivity
    .sort_values(
        "Wilcoxon_p_Linear_Holm"
    )
    .round(6)
)


,Feature,Mean_Delta_Linear,Rank_Biserial_Linear,Wilcoxon_p_Linear,Wilcoxon_p_Linear_Holm,Significant_Linear_Holm_0.05
6,Overlap_Pair_Fraction,-0.418854,-0.407921,0.000398,0.004377,True
5,Hyperdegree_Gini,-0.460084,-0.361188,0.001714,0.017141,True
7,Mean_Overlap_Jaccard,0.441167,0.356040,0.001994,0.017950,True
4,Hyperdegree_Concentration,-0.257230,-0.274851,0.017024,0.136189,False
10,Hypergraph_Spectral_Entropy,-0.326825,-0.273663,0.017507,0.136189,False
3,Max_Hyperdegree,-0.286698,-0.243960,0.034174,0.205046,False
9,Hypergraph_Lambda2,-0.207510,-0.237228,0.039441,0.205046,False
1,Max_Hyperedge_Weight,0.277950,0.189899,0.100915,0.403658,False
8,TwoSection_Density,0.185267,0.160808,0.164792,0.494375,False
2,Mean_Hyperdegree,0.052280,0.007921,0.945175,1.000000,False


## 16. Conservative hypergraph candidate screen

In [22]:
# ============================================================
# SECTION 16 — CONSERVATIVE HYPERGRAPH CANDIDATE SCREEN
# Self-contained replacement
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests


# ------------------------------------------------------------
# 1. Recreate paired word-count differences
# ------------------------------------------------------------

word_delta = (
    responses
    .pivot(
        index="Pair_ID",
        columns="Response_Type",
        values="Words"
    )
)

word_delta["Delta_Words"] = (
    word_delta["Hallucinated"]
    -
    word_delta["Factual"]
)


# ------------------------------------------------------------
# 2. Recreate delta_length_diagnostics
# ------------------------------------------------------------

delta_length_rows = []

for k in K_VALUES:

    subset = paired_normalized_differences[
        paired_normalized_differences["K"] == k
    ]

    for feature in HYPER_FEATURES:

        feature_delta = (
            subset[
                subset["Feature"] == feature
            ]
            .set_index("Pair_ID")["DeltaNorm"]
        )

        aligned = pd.concat(
            [
                feature_delta,
                word_delta["Delta_Words"]
            ],
            axis=1,
            join="inner"
        ).dropna()

        rho, p = spearmanr(
            aligned["DeltaNorm"],
            aligned["Delta_Words"]
        )

        delta_length_rows.append(
            {
                "K": int(k),
                "Feature": feature,
                "rho_DeltaNorm_vs_DeltaWords": rho,
                "p": p
            }
        )


delta_length_diagnostics = pd.DataFrame(
    delta_length_rows
)


# Holm correction separately for each k
for k in K_VALUES:

    mask = (
        delta_length_diagnostics["K"] == k
    )

    delta_length_diagnostics.loc[
        mask,
        "p_Holm"
    ] = multipletests(
        delta_length_diagnostics.loc[
            mask,
            "p"
        ],
        method="holm"
    )[1]


print(
    "delta_length_diagnostics recreated:",
    len(delta_length_diagnostics),
    "rows"
)


# ------------------------------------------------------------
# 3. Primary spline results for k = 3
# ------------------------------------------------------------

primary_spline = (
    paired_hypergraph_tests[
        paired_hypergraph_tests["K"] == PRIMARY_K
    ][
        [
            "Feature",
            "Mean_Normalized_Delta",
            "Rank_Biserial",
            "Wilcoxon_p_Holm",
            "Significant_Holm_0.05"
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# 4. Residual feature-vs-length correlation
# ------------------------------------------------------------

primary_length = (
    length_diagnostics[
        length_diagnostics["K"] == PRIMARY_K
    ][
        [
            "Feature",
            "Normalized_rho_with_Words"
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# 5. Delta-feature-vs-delta-length correlation
# ------------------------------------------------------------

primary_delta_length = (
    delta_length_diagnostics[
        delta_length_diagnostics["K"] == PRIMARY_K
    ][
        [
            "Feature",
            "rho_DeltaNorm_vs_DeltaWords"
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# 6. Merge all robustness information
# ------------------------------------------------------------

candidate_screen = (
    primary_spline

    .merge(
        linear_normalization_sensitivity,
        on="Feature",
        how="left"
    )

    .merge(
        primary_length,
        on="Feature",
        how="left"
    )

    .merge(
        primary_delta_length,
        on="Feature",
        how="left"
    )

    .merge(
        k_sensitivity[
            [
                "Feature",
                "Same_Direction_All_K",
                "Significant_All_K",
                "Primary_Signal_Stable_Direction"
            ]
        ],
        on="Feature",
        how="left"
    )
)


# ------------------------------------------------------------
# 7. Same direction under spline and linear normalization
# ------------------------------------------------------------

candidate_screen[
    "Same_Direction_Spline_Linear"
] = (
    np.sign(
        candidate_screen[
            "Mean_Normalized_Delta"
        ]
    )
    ==
    np.sign(
        candidate_screen[
            "Mean_Delta_Linear"
        ]
    )
)


# ------------------------------------------------------------
# 8. Weak residual dependence on response length
# ------------------------------------------------------------

candidate_screen[
    "Weak_Residual_Length_Association"
] = (
    candidate_screen[
        "Normalized_rho_with_Words"
    ].abs() < 0.20
)


# ------------------------------------------------------------
# 9. Weak paired dependence on Delta Words
# ------------------------------------------------------------

candidate_screen[
    "Weak_Paired_Length_Association"
] = (
    candidate_screen[
        "rho_DeltaNorm_vs_DeltaWords"
    ].abs() < 0.20
)


# ------------------------------------------------------------
# 10. Conservative candidate definition
# ------------------------------------------------------------

candidate_screen[
    "Conservative_Hypergraph_Candidate"
] = (

    candidate_screen[
        "Significant_Holm_0.05"
    ]

    &

    candidate_screen[
        "Significant_Linear_Holm_0.05"
    ]

    &

    candidate_screen[
        "Same_Direction_Spline_Linear"
    ]

    &

    candidate_screen[
        "Same_Direction_All_K"
    ]

    &

    candidate_screen[
        "Weak_Residual_Length_Association"
    ]

    &

    candidate_screen[
        "Weak_Paired_Length_Association"
    ]
)


# ------------------------------------------------------------
# 11. Display final candidate screen
# ------------------------------------------------------------

candidate_screen = (
    candidate_screen
    .sort_values(
        [
            "Conservative_Hypergraph_Candidate",
            "Wilcoxon_p_Holm"
        ],
        ascending=[
            False,
            True
        ]
    )
    .reset_index(drop=True)
)


print(
    "Conservative hypergraph candidate screen completed."
)

print(
    "Candidate features:",
    candidate_screen[
        "Conservative_Hypergraph_Candidate"
    ].sum()
)


display(
    candidate_screen.round(6)
)

delta_length_diagnostics recreated: 33 rows
Conservative hypergraph candidate screen completed.
Candidate features: 2


,Feature,Mean_Normalized_Delta,Rank_Biserial,Wilcoxon_p_Holm,Significant_Holm_0.05,Mean_Delta_Linear,Rank_Biserial_Linear,Wilcoxon_p_Linear,Wilcoxon_p_Linear_Holm,Significant_Linear_Holm_0.05,Normalized_rho_with_Words,rho_DeltaNorm_vs_DeltaWords,Same_Direction_All_K,Significant_All_K,Primary_Signal_Stable_Direction,Same_Direction_Spline_Linear,Weak_Residual_Length_Association,Weak_Paired_Length_Association,Conservative_Hypergraph_Candidate
0,Overlap_Pair_Fraction,-0.432051,-0.407129,0.004492,True,-0.418854,-0.407921,0.000398,0.004377,True,0.074901,-0.064158,True,False,True,True,True,True,True
1,Hyperdegree_Gini,-0.430254,-0.339010,0.032483,True,-0.460084,-0.361188,0.001714,0.017141,True,0.006410,-0.013003,True,False,True,True,True,True,True
2,Hyperdegree_Concentration,-0.260633,-0.265347,0.187717,False,-0.257230,-0.274851,0.017024,0.136189,False,0.058427,-0.029267,True,False,False,True,True,True,False
3,Mean_Overlap_Jaccard,0.363472,0.266139,0.187717,False,0.441167,0.356040,0.001994,0.017950,True,0.026921,0.081214,False,False,False,True,True,True,False
4,Hypergraph_Spectral_Entropy,-0.306694,-0.264554,0.187717,False,-0.326825,-0.273663,0.017507,0.136189,False,-0.001256,-0.108055,True,False,False,True,True,True,False
5,Max_Hyperedge_Weight,0.284975,0.201212,0.354446,False,0.277950,0.189899,0.100915,0.403658,False,0.076553,0.066860,True,False,False,True,True,True,False
6,Max_Hyperdegree,-0.271076,-0.217426,0.354446,False,-0.286698,-0.243960,0.034174,0.205046,False,-0.072233,-0.053676,True,False,False,True,True,True,False
7,Hypergraph_Lambda2,-0.183842,-0.209505,0.354446,False,-0.207510,-0.237228,0.039441,0.205046,False,0.112952,0.118507,True,False,False,True,True,True,False
8,Mean_Hyperedge_Weight,0.147773,0.069703,1.000000,False,0.126992,0.072871,0.526961,1.000000,False,-0.004132,-0.001243,True,False,False,True,True,True,False
9,Mean_Hyperdegree,0.099211,0.019406,1.000000,False,0.052280,0.007921,0.945175,1.000000,False,-0.004047,-0.031062,True,False,False,True,True,True,False


## 17. Primary \(k=3\) difficulty-level descriptive analysis

In [23]:

primary_pair_deltas = (
    paired_normalized_differences[
        paired_normalized_differences[
            "K"
        ] == PRIMARY_K
    ]
    .copy()
)

difficulty_map = (
    responses[
        [
            "Pair_ID",
            "Difficulty"
        ]
    ]
    .drop_duplicates(
        "Pair_ID"
    )
    .set_index(
        "Pair_ID"
    )[
        "Difficulty"
    ]
)

primary_pair_deltas[
    "Difficulty"
] = primary_pair_deltas[
    "Pair_ID"
].map(
    difficulty_map
)

difficulty_rows = []

for (
    difficulty,
    feature
), group in primary_pair_deltas.groupby(
    [
        "Difficulty",
        "Feature"
    ]
):

    delta = group[
        "DeltaNorm"
    ].astype(float)

    difficulty_rows.append(
        {
            "Difficulty":
                difficulty,

            "Feature":
                feature,

            "n":
                len(
                    group
                ),

            "Mean_Normalized_Delta":
                delta.mean(),

            "Median_Normalized_Delta":
                delta.median(),

            "Positive_n":
                int(
                    (
                        delta
                        >
                        0
                    ).sum()
                ),

            "Negative_n":
                int(
                    (
                        delta
                        <
                        0
                    ).sum()
                )
        }
    )

difficulty_summary = pd.DataFrame(
    difficulty_rows
)

display(
    difficulty_summary.round(
        6
    )
)


,Difficulty,Feature,n,Mean_Normalized_Delta,Median_Normalized_Delta,Positive_n,Negative_n
0,easy,Hyperdegree_Concentration,27,-0.438368,-0.758461,11,16
1,easy,Hyperdegree_Gini,27,-0.606847,-0.499161,10,17
2,easy,Hypergraph_Lambda2,27,-0.348519,-0.325438,8,19
3,easy,Hypergraph_Spectral_Entropy,27,-0.283633,-0.381598,10,17
4,easy,Max_Hyperdegree,27,-0.607778,-1.017053,11,16
5,easy,Max_Hyperedge_Weight,27,0.196796,-0.011654,12,14
6,easy,Mean_Hyperdegree,27,-0.096151,-0.257220,9,18
7,easy,Mean_Hyperedge_Weight,27,0.006377,-0.236549,10,17
8,easy,Mean_Overlap_Jaccard,27,0.601508,0.377902,15,12
9,easy,Overlap_Pair_Fraction,27,-0.685435,-0.953893,9,18


## 18. Redundancy among normalized hypergraph features

In [24]:

primary_delta_wide = (
    primary_pair_deltas
    .pivot(
        index="Pair_ID",
        columns="Feature",
        values="DeltaNorm"
    )
)

normalized_hypergraph_correlation = (
    primary_delta_wide
    .corr(
        method="spearman"
    )
)

display(
    normalized_hypergraph_correlation
    .round(
        3
    )
)


Feature,Hyperdegree_Concentration,Hyperdegree_Gini,Hypergraph_Lambda2,Hypergraph_Spectral_Entropy,Max_Hyperdegree,Max_Hyperedge_Weight,Mean_Hyperdegree,Mean_Hyperedge_Weight,Mean_Overlap_Jaccard,Overlap_Pair_Fraction,TwoSection_Density
Feature,,,,,,,,,,,
Hyperdegree_Concentration,1.000,0.751,0.425,0.549,0.893,0.343,0.235,0.271,-0.590,0.795,0.022
Hyperdegree_Gini,0.751,1.000,0.521,0.732,0.671,0.255,0.213,0.234,-0.639,0.859,0.068
Hypergraph_Lambda2,0.425,0.521,1.000,0.720,0.291,0.027,0.016,0.009,-0.642,0.611,0.429
Hypergraph_Spectral_Entropy,0.549,0.732,0.720,1.000,0.450,-0.013,0.048,0.022,-0.789,0.787,0.523
Max_Hyperdegree,0.893,0.671,0.291,0.450,1.000,0.407,0.454,0.478,-0.548,0.725,0.074
Max_Hyperedge_Weight,0.343,0.255,0.027,-0.013,0.407,1.000,0.545,0.666,-0.086,0.114,0.004
Mean_Hyperdegree,0.235,0.213,0.016,0.048,0.454,0.545,1.000,0.972,-0.104,0.169,0.042
Mean_Hyperedge_Weight,0.271,0.234,0.009,0.022,0.478,0.666,0.972,1.000,-0.114,0.167,0.046
Mean_Overlap_Jaccard,-0.590,-0.639,-0.642,-0.789,-0.548,-0.086,-0.104,-0.114,1.000,-0.828,-0.545


## 19. Explainability check: top primary-\(k=3\) hypergraph words

In [25]:

example_vertex_table = (
    primary_vertices[
        primary_vertices[
            "Pair_ID"
        ] == example_pair
    ]
    .sort_values(
        [
            "Response_Type",
            "Weighted_Hyperdegree"
        ],
        ascending=[
            True,
            False
        ]
    )
)

print(
    "MEDH_096 factual and hallucinated"
    " words ranked by weighted hyperdegree:"
)

display(
    example_vertex_table
    .groupby(
        "Response_Type",
        group_keys=False
    )
    .head(
        10
    )
)


MEDH_096 factual and hallucinated words ranked by weighted hyperdegree:


,Pair_ID,Response_Type,Node_ID,Word,Weighted_Hyperdegree,Hyperdegree_Share,Membership_Count
6251,MEDH_096,Factual,4,endothelin-1,0.257025,0.087406,9
6257,MEDH_096,Factual,10,aggravating,0.229551,0.078063,7
6263,MEDH_096,Factual,16,"pancreatitis,",0.199239,0.067755,6
6256,MEDH_096,Factual,9,in,0.175833,0.059795,5
6255,MEDH_096,Factual,8,role,0.172040,0.058505,5
6259,MEDH_096,Factual,12,development,0.153315,0.052137,4
6265,MEDH_096,Factual,18,its,0.146113,0.049688,4
6264,MEDH_096,Factual,17,through,0.131031,0.044559,4
6249,MEDH_096,Factual,2,suggest,0.130235,0.044289,5
6248,MEDH_096,Factual,1,results,0.128300,0.043631,5


## 20. Save Notebook 7 outputs

In [26]:

OUTPUT_DIR = (
    "/content/Notebook7_outputs"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

hypergraph_features.to_csv(
    f"{OUTPUT_DIR}/"
    "Hypergraph_Response_Features_AllK.csv",
    index=False
)

primary_hyperedges.to_csv(
    f"{OUTPUT_DIR}/"
    "Hypergraph_PrimaryK3_Edges.csv",
    index=False
)

primary_vertices.to_csv(
    f"{OUTPUT_DIR}/"
    "Hypergraph_PrimaryK3_Vertex_Metrics.csv",
    index=False
)

length_diagnostics.to_csv(
    f"{OUTPUT_DIR}/"
    "Hypergraph_Length_Normalization_Diagnostics.csv",
    index=False
)

normalized_hypergraphs.to_csv(
    f"{OUTPUT_DIR}/"
    "Hypergraph_Length_Normalized_Response_Features.csv",
    index=False
)

paired_hypergraph_tests.to_csv(
    f"{OUTPUT_DIR}/"
    "Hypergraph_Paired_Tests_AllK.csv",
    index=False
)

paired_normalized_differences.to_csv(
    f"{OUTPUT_DIR}/"
    "Hypergraph_Paired_Normalized_Differences.csv",
    index=False
)

delta_length_diagnostics.to_csv(
    f"{OUTPUT_DIR}/"
    "Hypergraph_Paired_Length_Diagnostics.csv",
    index=False
)

k_sensitivity.to_csv(
    f"{OUTPUT_DIR}/"
    "Hypergraph_K_Sensitivity.csv",
    index=False
)

linear_normalization_sensitivity.to_csv(
    f"{OUTPUT_DIR}/"
    "Hypergraph_Linear_Normalization_Sensitivity_K3.csv",
    index=False
)

candidate_screen.to_csv(
    f"{OUTPUT_DIR}/"
    "Hypergraph_Conservative_Candidate_Screen.csv",
    index=False
)

difficulty_summary.to_csv(
    f"{OUTPUT_DIR}/"
    "Hypergraph_Difficulty_Descriptives_K3.csv",
    index=False
)

normalized_hypergraph_correlation.to_csv(
    f"{OUTPUT_DIR}/"
    "Hypergraph_Normalized_Feature_Correlation_K3.csv"
)

print(
    "Notebook 7 outputs saved."
)


Notebook 7 outputs saved.


In [27]:

!zip -r \
/content/Notebook7_Hypergraph_Outputs.zip \
/content/Notebook7_outputs

files.download(
    "/content/Notebook7_Hypergraph_Outputs.zip"
)


  adding: content/Notebook7_outputs/ (stored 0%)
  adding: content/Notebook7_outputs/Hypergraph_PrimaryK3_Edges.csv (deflated 69%)
  adding: content/Notebook7_outputs/Hypergraph_Conservative_Candidate_Screen.csv (deflated 59%)
  adding: content/Notebook7_outputs/Hypergraph_Difficulty_Descriptives_K3.csv (deflated 57%)
  adding: content/Notebook7_outputs/Hypergraph_Normalized_Feature_Correlation_K3.csv (deflated 75%)
  adding: content/Notebook7_outputs/Hypergraph_Response_Features_AllK.csv (deflated 65%)
  adding: content/Notebook7_outputs/Hypergraph_Length_Normalization_Diagnostics.csv (deflated 51%)
  adding: content/Notebook7_outputs/Hypergraph_PrimaryK3_Vertex_Metrics.csv (deflated 62%)
  adding: content/Notebook7_outputs/Hypergraph_K_Sensitivity.csv (deflated 61%)
  adding: content/Notebook7_outputs/Hypergraph_Paired_Normalized_Differences.csv (deflated 64%)
  adding: content/Notebook7_outputs/Hypergraph_Linear_Normalization_Sensitivity_K3.csv (deflated 51%)
  adding: content/Noteb

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


## Interpretation rules for the paper

1. A raw hypergraph difference is **not** sufficient evidence because response length can alter incidence and spectral structure.
2. Primary inferential conclusions should use the **length-normalized \(k=3\)** results with Holm correction.
3. \(k=5\) and \(k=10\) are sensitivity analyses, not additional opportunities to select a favorable result.
4. A feature may be described as direction-stable only when its hallucinated-minus-factual direction is consistent across all tested \(k\) values.
5. Hypergraph spectral measures should be interpreted as structural properties of the attention-derived representation, not as causal explanations of the model's internal reasoning.
6. Vertex-level weighted hyperdegree is used for explainability/ranking only; high hyperdegree does not by itself mean that a word caused a hallucination.
7. Notebook 8 should combine only the most defensible graph and hypergraph signals with the fuzzy-risk layer, using pair-aware cross-validation to avoid leakage.
